In [ ]:
!pip install kagglehub[pandas-datasets]
!pip install ultralytics
from ultralytics import YOLO
import xml.etree.ElementTree as ET
import shutil
import kagglehub
import os

In [ ]:
path = kagglehub.dataset_download("dataclusterlabs/fire-and-smoke-dataset")
print(path)
print("Arquivos e pastas no diretório do dataset:", os.listdir(path))

Using Colab cache for faster access to the 'fire-and-smoke-dataset' dataset.
/kaggle/input/fire-and-smoke-dataset
Arquivos e pastas no diretório do dataset: ['Annotations', 'Datacluster Fire and Smoke Sample']


In [ ]:
images = os.path.join(path, 'Datacluster Fire and Smoke Sample', 'Datacluster Fire and Smoke Sample')
annotations = os.path.join(path, 'Annotations', 'Annotations')
print(os.listdir(images))
print(os.listdir(annotations))

['Datacluster Fire and Smoke Sample (15).jpg', 'Datacluster Fire and Smoke Sample (70).jpg', 'Datacluster Fire and Smoke Sample (22).jpg', 'Datacluster Fire and Smoke Sample (67).jpg', 'Datacluster Fire and Smoke Sample (66).jpg', 'Datacluster Fire and Smoke Sample (55).jpg', 'Datacluster Fire and Smoke Sample (46).jpg', 'Datacluster Fire and Smoke Sample (43).jpg', 'Datacluster Fire and Smoke Sample (84).jpg', 'Datacluster Fire and Smoke Sample (24).jpg', 'Datacluster Fire and Smoke Sample (21).jpg', 'Datacluster Fire and Smoke Sample (58).jpg', 'Datacluster Fire and Smoke Sample (86).jpg', 'Datacluster Fire and Smoke Sample (69).jpg', 'Datacluster Fire and Smoke Sample (75).jpg', 'Datacluster Fire and Smoke Sample (18).jpg', 'Datacluster Fire and Smoke Sample (12).jpg', 'Datacluster Fire and Smoke Sample (29).jpg', 'Datacluster Fire and Smoke Sample (32).jpg', 'Datacluster Fire and Smoke Sample (91).jpg', 'Datacluster Fire and Smoke Sample (26).jpg', 'Datacluster Fire and Smoke Sampl

In [ ]:
base_dir = 'dataset'
for folder in ['train', 'val']:
  os.makedirs(os.path.join(base_dir, folder, 'images'), exist_ok=True)
  os.makedirs(os.path.join(base_dir, folder, 'labels'), exist_ok=True)
class_map = {"fire": 0, "smoke": 1}
images_list = []
annotations_list = []
for img in os.listdir(images):
  if img.endswith('.jpg'):
    images_list.append(img)
print(f'total de imagens: {len(images_list)}')
cut = int(0.8 * (len(images_list)))
train = images_list[:cut]
val = images_list[cut:]
print(f"""
imagens para treino:{len(train)}
imagens para validação:{len(val)}""")

total de imagens: 100

imagens para treino:80
imagens para validação:20


In [ ]:
for img in train:
  origem =os.path.join(images,img)
  destino = os.path.join(base_dir, 'train', 'images', img)
  shutil.copy(origem, destino)
for img in val:
  origem =os.path.join(images,img)
  destino = os.path.join(base_dir, 'val', 'images', img)
  shutil.copy(origem, destino)

In [ ]:
# !cat '/root/.cache/kagglehub/datasets/dataclusterlabs/fire-and-smoke-dataset/versions/3/Annotations/Annotations/Datacluster Fire and Smoke Sample (12).xml'
for xml in os.listdir(annotations):
  linhas_txt = []
  caminho = os.path.join(annotations, xml)
  tree = ET.parse(caminho)
  root = tree.getroot()


  obj = root.find('object')
  if obj is None:
        print(f"{xml}")
        continue
  name = obj.find('name').text

  bndbox = obj.find('bndbox')
  xmin = float(bndbox.find('xmin').text)
  ymin = float(bndbox.find('ymin').text)
  xmax = float(bndbox.find('xmax').text)
  ymax = float(bndbox.find('ymax').text)

  size = root.find('size')
  w = float(size.find('width').text)
  h = float(size.find('height').text)

  class_id = class_map.get(name)
  x_center = (xmin + xmax) / 2 / w
  y_center = (ymin + ymax) / 2 / h
  width = (xmax - xmin) / w
  height = (ymax - ymin) / h

  # print(f"""{xmin} + {xmax} / 2 / {w} = {x_center} """)
  # Annotation line format: <class_id> <x_center> <y_center> <width> <height> de https://labelformat.com/formats/object-detection/yolov11/
  # print(class_id, x_center, y_center, width, height)
  linhas_txt.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
  nome_base, _ = os.path.splitext(xml)
  nome_img = nome_base + '.jpg'

  if nome_img in train:
      current_split = 'train'
  elif nome_img in val:
      current_split = 'val'
  else:
      continue

  caminho_txt = os.path.join(base_dir, current_split, 'labels', nome_base + '.txt')
  with open(caminho_txt, 'w') as f:
      f.write('\n'.join(linhas_txt))

Datacluster Fire and Smoke Sample (79).xml


In [ ]:
data = """path: /content/dataset
train: /content/dataset/train/images
val: /content/dataset/val/images

names:
  0: fire
  1: smoke"""
caminho = os.path.join(base_dir, 'dataset.yaml')
with open(caminho, 'w') as f:
  f.write(data)

In [ ]:
!yolo task=detect mode=train model=yolo11n.pt data=/content/dataset/dataset.yaml epochs=100 imgsz=960

Ultralytics 8.4.178 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/dataset.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-3, nbs=64, nms=None, opset=None, optimize=False, opti

In [ ]:
!yolo detect predict model=/content/runs/detect/train-3/weights/best.pt source=/content/a.mp4 save=True